# Scenario Runner

Live scenario reset, parent-order execution, venue fill breakdown, and run controls for the SOR engine API.

Edit these values before running the next cell. `quantity`, `max_route_attempts`, and `route_timeout_millis` control how the engine reslices until the order is filled or the timeout/attempt limit is reached.

In [ ]:
from IPython.display import HTML, display
import importlib
import pandas as pd
import adaptive_quantum_sor.client as sor_client_module

sor_client_module = importlib.reload(sor_client_module)
SorNotebookClient = sor_client_module.SorNotebookClient

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)

def display_wrapped(frame):
    if frame is None or frame.empty:
        display(frame)
        return
    html_table = frame.to_html(index=False, escape=True)
    display(HTML("""
    <style>
    table.sor-wrap-table {border-collapse:collapse; width:100%; table-layout:auto; margin:8px 0 18px 0;}
    table.sor-wrap-table th, table.sor-wrap-table td {
        border:1px solid #d6dae1; padding:6px 8px; vertical-align:top;
        white-space:normal; overflow-wrap:anywhere; word-break:break-word;
        max-width:520px; text-align:left;
    }
    table.sor-wrap-table th {background:#f8fafc; color:#111827;}
    </style>
    """ + html_table.replace('class="dataframe"', 'class="sor-wrap-table"')))

client = SorNotebookClient()

scenario_id = 'notebook-scenario'
seed = 42
ticks = 10
reset_mode = 'PURGE_AND_REPOPULATE'

instrument_id = 0
side = 'BUY'  # BUY or SELL; numeric 1/2 is also accepted
quantity = 2500
urgency_id = 0
at_tick = 1
submit_mode = 'SIMULATED'  # SIMULATED or API
client_order_ref = 'notebook-parent-1'
max_route_attempts = 8
route_timeout_millis = 1000

parent_order = {
    'instrumentId': instrument_id,
    'side': side,
    'quantity': quantity,
    'urgencyId': urgency_id,
    'atTick': at_tick,
    'submitMode': submit_mode,
    'clientOrderRef': client_order_ref,
}

display_wrapped(pd.DataFrame([{
    'scenarioId': scenario_id,
    'seed': seed,
    'ticks': ticks,
    'resetMode': reset_mode,
    'maxRouteAttempts': max_route_attempts,
    'routeTimeoutMillis': route_timeout_millis,
    **parent_order,
}]))


In [ ]:
reset = client.reset_scenario(scenario_id, seed=seed, ticks=ticks, reset_mode=reset_mode)
run = client.run_scenario(
    scenario_id,
    seed=seed,
    ticks=ticks,
    reset_mode=reset_mode,
    parent_orders=[parent_order],
    max_route_attempts=max_route_attempts,
    route_timeout_millis=route_timeout_millis,
    simulator_generated_orders=False,
)
summary = client.scenario_summary_dataframe()
events = client.scenario_events_dataframe()

parent_order_results = pd.DataFrame([
    {key: value for key, value in result.items() if key != 'fills'}
    for result in run.get('parentOrderResults', [])
])
fill_rows = []
for result in run.get('parentOrderResults', []):
    for fill in result.get('fills', []):
        fill_rows.append(fill)
fills = pd.DataFrame(fill_rows)

display_wrapped(pd.DataFrame([reset]).add_prefix('reset.'))
display_wrapped(pd.DataFrame([run]).drop(columns=['parentOrderResults'], errors='ignore').add_prefix('run.'))
display_wrapped(parent_order_results.add_prefix('order.'))
display_wrapped(fills.add_prefix('fill.'))
display_wrapped(summary.add_prefix('summary.'))
event_columns = ['sequence', 'eventId', 'timestampNanos', 'event', 'componentId', 'eventType', 'correlationId', 'message']
display_wrapped(events[[column for column in event_columns if column in events.columns]])
